# IEEE 33-Bus Graph Transformer Dataset Pipeline (HDF5)

This notebook builds a reproducible, scenario-based power-flow dataset for the IEEE 33-bus distribution system. Each sample combines load perturbations and a feasible radial switch configuration, solves power flow, extracts graph features, and streams the result to HDF5 for Graph Transformer research.

**Research targets**
- Active/reactive load variation impact prediction
- Switching behavior modeling
- Contingency and reconfiguration analysis
- Graph Transformer labels: losses, voltage profile, and line loading

**Storage target**: `data/ieee33_dataset.h5`

In [ ]:
from pathlib import Path
import copy
import json

import pandapower as pp
import pandapower.networks as pn
import networkx as nx
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
import h5py
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

GLOBAL_SEED = 20260622
N_SAMPLES = 1000
BATCH_SIZE = 64
N_JOBS = -1
H5_PATH = Path("data/ieee33_dataset.h5")
COMPRESSION = "gzip"
COMPRESSION_OPTS = 4
VOLTAGE_LIMITS_PU = (0.95, 1.05)
OVERLOAD_LIMIT_PERCENT = 100.0

np.random.seed(GLOBAL_SEED)
plt.rcParams["figure.figsize"] = (9, 5)

print(f"Configuration ready | samples={N_SAMPLES:,} | batch_size={BATCH_SIZE} | seed={GLOBAL_SEED}")

## 1. IEEE-33 Network Loading and Switchable Tie-Line Preparation

`pandapower.networks.case33bw()` is the authoritative base network. The helper below preserves the base radial feeder and ensures the five standard IEEE-33 normally-open tie branches are represented as switchable out-of-service lines. This makes branch-exchange sampling deterministic and explicit.

In [ ]:
# Standard IEEE-33 normally-open tie branches, zero-based bus numbering.
# Values are the conventional Baran-Wu tie branch impedances used when the
# pandapower case representation does not already include these branches.
STANDARD_TIE_BRANCH_DATA = [
    {"from_bus": 20, "to_bus": 7, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 8, "to_bus": 14, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 11, "to_bus": 21, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 17, "to_bus": 32, "r_ohm": 0.5, "x_ohm": 0.5},
    {"from_bus": 24, "to_bus": 28, "r_ohm": 0.5, "x_ohm": 0.5},
]


def _ensure_line_columns(net):
    """Ensure line status and tie markers exist with stable boolean semantics."""
    if "in_service" not in net.line.columns:
        net.line["in_service"] = True
    net.line["in_service"] = net.line["in_service"].fillna(True).astype(bool)

    if "is_tie" not in net.line.columns:
        net.line["is_tie"] = False
    net.line["is_tie"] = net.line["is_tie"].fillna(False).astype(bool)


def _line_mask_for_endpoints(net, from_bus, to_bus):
    from_values = net.line["from_bus"].astype(int).to_numpy()
    to_values = net.line["to_bus"].astype(int).to_numpy()
    return ((from_values == int(from_bus)) & (to_values == int(to_bus))) | ((from_values == int(to_bus)) & (to_values == int(from_bus)))


def _median_or_default(series, default_value):
    if series is None or len(series) == 0:
        return float(default_value)
    value = float(pd.Series(series).dropna().median())
    return value if np.isfinite(value) else float(default_value)


def _ensure_ieee33_tie_lines(net):
    """Add or mark the five standard IEEE-33 tie branches as normally open."""
    _ensure_line_columns(net)
    max_i_ka = _median_or_default(net.line["max_i_ka"] if "max_i_ka" in net.line.columns else None, 0.4)
    c_nf_per_km = _median_or_default(net.line["c_nf_per_km"] if "c_nf_per_km" in net.line.columns else None, 0.0)

    for tie in STANDARD_TIE_BRANCH_DATA:
        mask = _line_mask_for_endpoints(net, tie["from_bus"], tie["to_bus"])
        matching_indices = list(net.line.index[mask])

        if matching_indices:
            net.line.loc[matching_indices, "is_tie"] = True
            net.line.loc[matching_indices, "in_service"] = False
            continue

        line_idx = pp.create_line_from_parameters(
            net,
            from_bus=tie["from_bus"],
            to_bus=tie["to_bus"],
            length_km=1.0,
            r_ohm_per_km=tie["r_ohm"],
            x_ohm_per_km=tie["x_ohm"],
            c_nf_per_km=c_nf_per_km,
            max_i_ka=max_i_ka,
            name=f"tie_{tie['from_bus'] + 1}_{tie['to_bus'] + 1}",
            in_service=False,
        )
        net.line.at[line_idx, "is_tie"] = True

    net.line["in_service"] = net.line["in_service"].astype(bool)
    net.line["is_tie"] = net.line["is_tie"].astype(bool)
    return net


def get_root_bus(net):
    """Return the substation/root bus from ext_grid when available."""
    if len(net.ext_grid) > 0 and "bus" in net.ext_grid.columns:
        return int(net.ext_grid.iloc[0]["bus"])
    return int(net.bus.index.min())


def load_ieee33():
    """Load IEEE-33, prepare switchable ties, and return pandapower net plus base graph."""
    net = pn.case33bw()
    _ensure_ieee33_tie_lines(net)
    base_graph = build_graph(net, active_only=True)
    if not validate_configuration(net):
        raise ValueError("Prepared IEEE-33 base configuration is not radial and connected.")
    return net, base_graph

## 2. Graph Builder and Topology Features

The graph builder supports two modes:
- `active_only=True` for physical feasibility checks on the currently energized radial feeder.
- `active_only=False` for feature extraction over all physical branches, where `switch_status` identifies open tie or sectionalizing branches.

In [ ]:
def _bus_load_table(net):
    """Aggregate active load at each bus in MW/Mvar."""
    loads = pd.DataFrame(index=net.bus.index, data={"p_load": 0.0, "q_load": 0.0})
    if len(net.load) == 0:
        return loads

    load_df = net.load.copy()
    if "in_service" in load_df.columns:
        load_df = load_df[load_df["in_service"].fillna(True).astype(bool)]
    if len(load_df) == 0:
        return loads

    grouped = load_df.groupby("bus")[["p_mw", "q_mvar"]].sum()
    loads.loc[grouped.index, "p_load"] = grouped["p_mw"].astype(float)
    loads.loc[grouped.index, "q_load"] = grouped["q_mvar"].astype(float)
    return loads


def _initial_voltage_pu_table(net):
    """Flat-start voltage profile, using ext_grid setpoints at substation buses."""
    voltage = pd.Series(1.0, index=net.bus.index, dtype=float)
    if len(net.ext_grid) > 0:
        for _, row in net.ext_grid.iterrows():
            vm_pu = float(row["vm_pu"]) if "vm_pu" in net.ext_grid.columns else 1.0
            voltage.loc[int(row["bus"])] = vm_pu
    return voltage


def line_is_closed(net, line_idx):
    """Return whether a line is electrically closed, combining line and switch status."""
    closed = bool(net.line.at[line_idx, "in_service"])
    if "switch" in net and len(net.switch) > 0:
        switches = net.switch[(net.switch["et"] == "l") & (net.switch["element"].astype(int) == int(line_idx))]
        if len(switches) > 0:
            closed = closed and bool(switches["closed"].fillna(True).astype(bool).all())
    return bool(closed)


def _line_impedance(net, line_idx):
    row = net.line.loc[line_idx]
    length = float(row["length_km"]) if "length_km" in net.line.columns else 1.0
    r = float(row["r_ohm_per_km"]) * length if "r_ohm_per_km" in net.line.columns else 0.0
    x = float(row["x_ohm_per_km"]) * length if "x_ohm_per_km" in net.line.columns else 0.0
    return r, x


def build_graph(net, active_only=False):
    """Convert a pandapower network to a NetworkX graph with node and edge features."""
    _ensure_line_columns(net)
    G = nx.Graph()
    load_table = _bus_load_table(net)
    voltage_table = _initial_voltage_pu_table(net)
    root_bus = get_root_bus(net)

    for bus_idx in net.bus.index:
        bus_idx_int = int(bus_idx)
        G.add_node(
            bus_idx_int,
            p_load=float(load_table.loc[bus_idx, "p_load"]),
            q_load=float(load_table.loc[bus_idx, "q_load"]),
            bus_type=1.0 if bus_idx_int == root_bus else 0.0,
            voltage=float(voltage_table.loc[bus_idx]),
        )

    for line_idx, row in net.line.iterrows():
        switch_status = 1.0 if line_is_closed(net, line_idx) else 0.0
        if active_only and switch_status < 0.5:
            continue
        r, x = _line_impedance(net, line_idx)
        G.add_edge(
            int(row["from_bus"]),
            int(row["to_bus"]),
            line_idx=int(line_idx),
            r=float(r),
            x=float(x),
            impedance=float(np.sqrt(r * r + x * x)),
            switch_status=float(switch_status),
            is_tie=bool(row["is_tie"]),
        )
    return G


def compute_feeder_depth(G, root_bus):
    """Compute BFS feeder depth from the substation/root bus."""
    depth = {int(node): -1 for node in G.nodes}
    if root_bus not in G.nodes:
        return depth
    for node, value in nx.single_source_shortest_path_length(G, int(root_bus)).items():
        depth[int(node)] = int(value)
    return depth


def compute_distance_to_substation(G, root_bus):
    """Compute impedance-weighted shortest-path distance to the substation/root bus."""
    distance = {int(node): np.inf for node in G.nodes}
    if root_bus not in G.nodes:
        return distance
    try:
        lengths = nx.single_source_dijkstra_path_length(G, int(root_bus), weight="impedance")
    except Exception:
        lengths = nx.single_source_shortest_path_length(G, int(root_bus))
    for node, value in lengths.items():
        distance[int(node)] = float(value)
    return distance

## 3. Load Perturbations, Radiality Checks, and Branch-Exchange Switching

Each sample independently scales bus loads and then performs a single feasible branch exchange: close one normally-open tie line and open one branch on the induced cycle. Invalid configurations are discarded before any power-flow solve.

In [ ]:
def check_connectivity(G):
    """Return True when all buses are connected in the active graph."""
    return len(G.nodes) > 0 and nx.is_connected(G)


def check_radiality(G):
    """Return True when the active graph is radial. For connected graphs this is nx.is_tree."""
    return len(G.nodes) > 0 and nx.is_tree(G)


def validate_configuration(net):
    """Validate the energized feeder topology independently of the power-flow solver."""
    active_graph = build_graph(net, active_only=True)
    return check_connectivity(active_graph) and check_radiality(active_graph)


def perturb_loads(net, min_scale=0.8, max_scale=1.2, rng=None):
    """Scale each bus load independently and return per-bus scaling factors."""
    if rng is None:
        rng = np.random.default_rng(GLOBAL_SEED)

    scales = pd.Series(1.0, index=net.bus.index, dtype=float)
    if len(net.load) == 0:
        return scales.to_numpy(dtype=np.float32)

    load_buses = sorted(pd.unique(net.load["bus"].astype(int)))
    for bus_idx in load_buses:
        scales.loc[bus_idx] = float(rng.uniform(min_scale, max_scale))

    for load_idx, row in net.load.iterrows():
        scale = float(scales.loc[int(row["bus"])])
        net.load.at[load_idx, "p_mw"] = float(row["p_mw"]) * scale
        net.load.at[load_idx, "q_mvar"] = float(row["q_mvar"]) * scale

    return scales.to_numpy(dtype=np.float32)


def _switch_configuration_key(net):
    """Stable key from active line indices for uniqueness analysis."""
    active_lines = [str(int(idx)) for idx in net.line.index if line_is_closed(net, idx)]
    return ",".join(active_lines)


def _cycle_edges_from_nodes(net, cycle_nodes):
    cycle_set = set(int(node) for node in cycle_nodes)
    cycle_edges = []
    for line_idx, row in net.line.iterrows():
        if not line_is_closed(net, line_idx):
            continue
        if int(row["from_bus"]) in cycle_set and int(row["to_bus"]) in cycle_set:
            cycle_edges.append(int(line_idx))
    return cycle_edges


def generate_switch_configuration(net, rng=None, max_attempts=100):
    """Generate a feasible branch-exchange configuration.

    The sampler closes one normally-open tie line, finds the resulting cycle,
    opens one non-tie branch on that cycle, and validates that the final active
    topology is connected and radial.
    """
    if rng is None:
        rng = np.random.default_rng(GLOBAL_SEED)

    _ensure_ieee33_tie_lines(net)
    original_status = net.line["in_service"].astype(bool).copy()
    tie_lines = [int(idx) for idx in net.line.index[net.line["is_tie"].astype(bool)]]
    inactive_ties = [idx for idx in tie_lines if not line_is_closed(net, idx)]

    if len(inactive_ties) == 0:
        raise ValueError("No inactive tie lines are available for branch exchange.")

    for attempt in range(max_attempts):
        net.line["in_service"] = original_status.copy()

        closed_tie = int(rng.choice(inactive_ties))
        net.line.at[closed_tie, "in_service"] = True

        trial_graph = build_graph(net, active_only=True)
        cycles = nx.cycle_basis(trial_graph)
        if len(cycles) == 0:
            continue

        closed_tie_row = net.line.loc[closed_tie]
        tie_endpoints = {int(closed_tie_row["from_bus"]), int(closed_tie_row["to_bus"])}
        selected_cycle = None
        for cycle in cycles:
            if tie_endpoints.issubset(set(int(node) for node in cycle)):
                selected_cycle = cycle
                break
        if selected_cycle is None:
            selected_cycle = cycles[0]

        cycle_edges = _cycle_edges_from_nodes(net, selected_cycle)
        open_candidates = [idx for idx in cycle_edges if idx != closed_tie and not bool(net.line.at[idx, "is_tie"])]
        if len(open_candidates) == 0:
            continue

        opened_branch = int(rng.choice(open_candidates))
        net.line.at[opened_branch, "in_service"] = False

        if validate_configuration(net):
            opened_row = net.line.loc[opened_branch]
            summary = {
                "closed_tie_line": int(closed_tie),
                "closed_tie_edge": [int(closed_tie_row["from_bus"]), int(closed_tie_row["to_bus"])],
                "opened_branch_line": int(opened_branch),
                "opened_branch_edge": [int(opened_row["from_bus"]), int(opened_row["to_bus"])],
                "attempts": int(attempt + 1),
                "config_key": _switch_configuration_key(net),
            }
            return summary

    net.line["in_service"] = original_status.copy()
    raise ValueError(f"Failed to generate a feasible branch exchange after {max_attempts} attempts.")

## 4. Power Flow Engine and Graph Feature Extraction

Power-flow failures are returned as structured failures. Feature extraction uses the active topology only for depth/distance calculations and stores all physical line features with switch status.

In [ ]:
NODE_FEATURE_NAMES = ["p_load_mw", "q_load_mvar", "depth", "distance_to_substation"]
EDGE_FEATURE_NAMES = ["r_ohm", "x_ohm", "switch_status"]
GLOBAL_FEATURE_NAMES = ["total_load_mw", "num_buses", "num_lines"]
LABEL_NAMES = ["total_loss_kw", "min_voltage_pu", "max_loading_percent", "violations"]


def run_powerflow(net):
    """Run pandapower power flow and return robust scalar labels."""
    try:
        pp.runpp(
            net,
            algorithm="nr",
            init="flat",
            calculate_voltage_angles=False,
            enforce_q_lims=False,
            max_iteration=50,
            tolerance_mva=1e-8,
            numba=False,
        )
    except Exception as exc:
        return {"success": False, "reason": f"powerflow_failed: {type(exc).__name__}: {exc}"}

    if not bool(getattr(net, "converged", False)):
        return {"success": False, "reason": "powerflow_not_converged"}

    total_loss_kw = 0.0
    if "res_line" in net and len(net.res_line) > 0 and "pl_mw" in net.res_line.columns:
        total_loss_kw += float(np.nan_to_num(net.res_line["pl_mw"].to_numpy(dtype=float), nan=0.0).sum() * 1000.0)
    if "res_trafo" in net and len(net.res_trafo) > 0 and "pl_mw" in net.res_trafo.columns:
        total_loss_kw += float(np.nan_to_num(net.res_trafo["pl_mw"].to_numpy(dtype=float), nan=0.0).sum() * 1000.0)

    vm_pu = np.nan_to_num(net.res_bus["vm_pu"].to_numpy(dtype=float), nan=np.nan)
    loading = np.array([0.0], dtype=float)
    if "res_line" in net and len(net.res_line) > 0 and "loading_percent" in net.res_line.columns:
        loading = np.nan_to_num(net.res_line["loading_percent"].to_numpy(dtype=float), nan=0.0, posinf=0.0, neginf=0.0)

    v_min, v_max = VOLTAGE_LIMITS_PU
    voltage_violation_count = int(((vm_pu < v_min) | (vm_pu > v_max)).sum())
    overload_count = int((loading > OVERLOAD_LIMIT_PERCENT).sum())

    return {
        "success": True,
        "total_loss_kw": float(total_loss_kw),
        "min_voltage_pu": float(np.nanmin(vm_pu)),
        "max_line_loading_percent": float(np.nanmax(loading)),
        "voltage_violation_count": voltage_violation_count,
        "overload_count": overload_count,
        "violations": int(voltage_violation_count + overload_count),
    }


def extract_graph_features(net, G=None):
    """Extract node, edge, and global feature matrices for one solved scenario."""
    if G is None:
        G = build_graph(net, active_only=False)
    active_graph = build_graph(net, active_only=True)
    root_bus = get_root_bus(net)
    depth = compute_feeder_depth(active_graph, root_bus)
    distance = compute_distance_to_substation(active_graph, root_bus)
    load_table = _bus_load_table(net)

    node_features = []
    for bus_idx in net.bus.index:
        bus_idx_int = int(bus_idx)
        node_features.append([
            float(load_table.loc[bus_idx, "p_load"]),
            float(load_table.loc[bus_idx, "q_load"]),
            float(depth.get(bus_idx_int, -1)),
            float(distance.get(bus_idx_int, np.inf)),
        ])

    edge_features = []
    for line_idx, _ in net.line.iterrows():
        r, x = _line_impedance(net, line_idx)
        edge_features.append([
            float(r),
            float(x),
            1.0 if line_is_closed(net, line_idx) else 0.0,
        ])

    total_load = float(load_table["p_load"].sum())
    global_features = np.array([
        total_load,
        float(len(net.bus)),
        float(len(net.line)),
    ], dtype=np.float32)

    return {
        "node_features": np.asarray(node_features, dtype=np.float32),
        "edge_features": np.asarray(edge_features, dtype=np.float32),
        "global_features": global_features,
    }

## 5. Single-Sample Generator and Streaming HDF5 Storage

Workers generate compact per-sample arrays and metadata. HDF5 writes are serialized in the notebook process to avoid concurrent file access issues.

In [ ]:
BASE_NET = None
BASE_GRAPH = None


def initialize_base_network():
    """Initialize global base objects once for reproducible sample generation."""
    global BASE_NET, BASE_GRAPH
    BASE_NET, BASE_GRAPH = load_ieee33()
    print(
        "Base IEEE-33 ready | "
        f"buses={len(BASE_NET.bus)} | lines={len(BASE_NET.line)} | "
        f"active_lines={int(BASE_NET.line['in_service'].sum())} | ties={int(BASE_NET.line['is_tie'].sum())}"
    )
    return BASE_NET, BASE_GRAPH


def _copy_base_network():
    """Deep-copy the base network without mutating BASE_NET."""
    if BASE_NET is None:
        initialize_base_network()
    return copy.deepcopy(BASE_NET)


def generate_sample(seed):
    """Generate one scenario: perturb loads, reconfigure switches, validate, solve, extract."""
    rng = np.random.default_rng(int(seed))
    try:
        net = _copy_base_network()
        load_scaling = perturb_loads(net, min_scale=0.8, max_scale=1.2, rng=rng)
        switch_summary = generate_switch_configuration(net, rng=rng)

        if not validate_configuration(net):
            return {"success": False, "seed": int(seed), "reason": "invalid_configuration"}

        pf_result = run_powerflow(net)
        if not pf_result["success"]:
            return {"success": False, "seed": int(seed), "reason": pf_result["reason"]}

        graph = build_graph(net, active_only=False)
        features = extract_graph_features(net, graph)
        labels = np.array([
            pf_result["total_loss_kw"],
            pf_result["min_voltage_pu"],
            pf_result["max_line_loading_percent"],
            pf_result["violations"],
        ], dtype=np.float32)

        metadata = {
            "seed": int(seed),
            "load_scaling_factors": [float(value) for value in load_scaling],
            "switch_action_summary": switch_summary,
            "switch_config_key": switch_summary["config_key"],
            "voltage_violation_count": int(pf_result["voltage_violation_count"]),
            "overload_count": int(pf_result["overload_count"]),
        }

        return {
            "success": True,
            "seed": int(seed),
            "node_features": features["node_features"],
            "edge_features": features["edge_features"],
            "global_features": features["global_features"],
            "labels": labels,
            "metadata": metadata,
        }
    except Exception as exc:
        return {"success": False, "seed": int(seed), "reason": f"sample_failed: {type(exc).__name__}: {exc}"}


def initialize_hdf5(path, overwrite=True):
    """Create an HDF5 file and write dataset-level metadata."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    mode = "w" if overwrite else "a"
    with h5py.File(path, mode, track_order=True) as h5:
        h5.attrs["schema_version"] = "1.0"
        h5.attrs["dataset_name"] = "ieee33_graph_transformer_dataset"
        h5.attrs["global_seed"] = int(GLOBAL_SEED)
        h5.attrs["node_feature_names"] = json.dumps(NODE_FEATURE_NAMES)
        h5.attrs["edge_feature_names"] = json.dumps(EDGE_FEATURE_NAMES)
        h5.attrs["global_feature_names"] = json.dumps(GLOBAL_FEATURE_NAMES)
        h5.attrs["label_names"] = json.dumps(LABEL_NAMES)
        h5.attrs["total_attempted"] = 0
        h5.attrs["valid_samples"] = 0
        h5.attrs["rejected_samples"] = 0
        h5.attrs["rejection_reasons"] = json.dumps({})
    return path


def _write_array(group, name, value):
    group.create_dataset(
        name,
        data=np.asarray(value),
        compression=COMPRESSION,
        compression_opts=COMPRESSION_OPTS,
        shuffle=True,
    )


def write_records_to_hdf5(path, records, start_sample_id):
    """Append valid records to HDF5 with one group per sample."""
    string_dtype = h5py.string_dtype(encoding="utf-8")
    next_id = int(start_sample_id)
    with h5py.File(path, "a", track_order=True) as h5:
        for record in records:
            group_name = f"sample_{next_id:06d}"
            group = h5.create_group(group_name)
            _write_array(group, "node_features", record["node_features"].astype(np.float32))
            _write_array(group, "edge_features", record["edge_features"].astype(np.float32))
            _write_array(group, "global_features", record["global_features"].astype(np.float32))
            _write_array(group, "labels", record["labels"].astype(np.float32))
            group.create_dataset("metadata", data=json.dumps(record["metadata"]), dtype=string_dtype)
            group.attrs["seed"] = int(record["seed"])
            group.attrs["switch_config_key"] = record["metadata"]["switch_config_key"]
            group.attrs["total_load_mw"] = float(record["global_features"][0])
            next_id += 1
    return next_id


def update_hdf5_report(path, total_attempted, valid_samples, rejected_samples, rejection_reasons):
    with h5py.File(path, "a") as h5:
        h5.attrs["total_attempted"] = int(total_attempted)
        h5.attrs["valid_samples"] = int(valid_samples)
        h5.attrs["rejected_samples"] = int(rejected_samples)
        h5.attrs["rejection_reasons"] = json.dumps(rejection_reasons)

## 6. Parallel Dataset Generation

Default generation creates `N_SAMPLES = 1000` scenarios. To scale later, increase `N_SAMPLES` (for example to `100_000`) and tune `BATCH_SIZE`/`N_JOBS` for the host CPU. The pipeline writes valid samples immediately after each batch and never holds the full graph dataset in RAM.

In [ ]:
def _seed_sequence(n_samples, global_seed=GLOBAL_SEED):
    return [int(global_seed + i) for i in range(int(n_samples))]


def generate_dataset(n_samples=N_SAMPLES, h5_path=H5_PATH, batch_size=BATCH_SIZE, n_jobs=N_JOBS, overwrite=True):
    """Generate a complete HDF5 dataset with process-safe single-writer batching."""
    initialize_base_network()
    h5_path = initialize_hdf5(h5_path, overwrite=overwrite)
    seeds = _seed_sequence(n_samples, GLOBAL_SEED)

    total_attempted = 0
    valid_samples = 0
    rejected_samples = 0
    rejection_reasons = {}
    next_sample_id = 1

    progress = tqdm(total=n_samples, desc="Generating IEEE-33 samples", unit="sample")
    for start in range(0, n_samples, batch_size):
        batch_seeds = seeds[start:start + batch_size]
        results = Parallel(n_jobs=n_jobs, backend="loky")(
            delayed(generate_sample)(seed) for seed in batch_seeds
        )

        valid_records = []
        for result in results:
            total_attempted += 1
            if result.get("success", False):
                valid_records.append(result)
            else:
                rejected_samples += 1
                reason = result.get("reason", "unknown")
                rejection_reasons[reason] = rejection_reasons.get(reason, 0) + 1

        if valid_records:
            next_sample_id = write_records_to_hdf5(h5_path, valid_records, next_sample_id)
            valid_samples += len(valid_records)

        update_hdf5_report(h5_path, total_attempted, valid_samples, rejected_samples, rejection_reasons)
        progress.update(len(batch_seeds))
    progress.close()

    success_rate = 100.0 * valid_samples / max(total_attempted, 1)
    report = {
        "h5_path": str(h5_path),
        "total_attempted": int(total_attempted),
        "valid_samples": int(valid_samples),
        "rejected_samples": int(rejected_samples),
        "success_rate_percent": float(success_rate),
        "rejection_reasons": rejection_reasons,
    }
    print(json.dumps(report, indent=2))
    return report


# Run the production default dataset generation.
generation_report = generate_dataset(N_SAMPLES, H5_PATH, BATCH_SIZE, N_JOBS, overwrite=True)

## 7. Streaming Dataset Analysis with Plotly

The analysis reader scans HDF5 sample groups and collects scalar labels, global features, switch keys, and load scaling factors. It does not load node or edge feature matrices into memory.

In [ ]:
def _read_metadata_dataset(dataset):
    raw = dataset[()]
    if isinstance(raw, bytes):
        raw = raw.decode("utf-8")
    return json.loads(raw)


def collect_dataset_analysis(h5_path=H5_PATH):
    """Collect scalar summaries for interactive analysis without loading graph arrays."""
    rows = []
    load_scales = []
    attrs = {}

    with h5py.File(h5_path, "r") as h5:
        attrs = {key: h5.attrs[key] for key in h5.attrs.keys()}
        sample_names = sorted(name for name in h5.keys() if name.startswith("sample_"))
        for sample_name in tqdm(sample_names, desc="Reading scalar summaries", unit="sample"):
            group = h5[sample_name]
            labels = group["labels"][:]
            global_features = group["global_features"][:]
            metadata = _read_metadata_dataset(group["metadata"])
            load_scales.extend(metadata.get("load_scaling_factors", []))
            rows.append({
                "sample": sample_name,
                "seed": int(metadata["seed"]),
                "total_load_mw": float(global_features[0]),
                "total_loss_kw": float(labels[0]),
                "min_voltage_pu": float(labels[1]),
                "max_loading_percent": float(labels[2]),
                "violations": int(labels[3]),
                "voltage_violation_count": int(metadata.get("voltage_violation_count", 0)),
                "overload_count": int(metadata.get("overload_count", 0)),
                "switch_config_key": metadata.get("switch_config_key", ""),
            })

    analysis_df = pd.DataFrame(rows)
    load_scale_series = pd.Series(load_scales, name="load_scaling_factor", dtype=float)
    return analysis_df, load_scale_series, attrs


def print_feasibility_report(attrs):
    total = int(attrs.get("total_attempted", 0))
    valid = int(attrs.get("valid_samples", 0))
    rejected = int(attrs.get("rejected_samples", 0))
    success_rate = 100.0 * valid / max(total, 1)
    reasons = json.loads(attrs.get("rejection_reasons", "{}"))
    print("Feasibility Report")
    print("-" * 22)
    print(f"Total samples attempted : {total:,}")
    print(f"Valid samples           : {valid:,}")
    print(f"Rejected samples        : {rejected:,}")
    print(f"Success rate            : {success_rate:.2f}%")
    print(f"Rejection reasons       : {json.dumps(reasons, indent=2)}")


def plot_loss_distribution(df):
    fig = px.histogram(df, x="total_loss_kw", nbins=50, title="Total Loss Distribution", labels={"total_loss_kw": "Total loss (kW)"})
    fig.update_layout(bargap=0.02)
    fig.show()


def plot_voltage_distribution(df):
    hist = px.histogram(df, x="min_voltage_pu", nbins=50, title="Minimum Voltage Distribution", labels={"min_voltage_pu": "Minimum voltage (pu)"})
    hist.update_layout(bargap=0.02)
    hist.show()

    box = px.box(df, y="min_voltage_pu", title="Minimum Voltage Box Plot", labels={"min_voltage_pu": "Minimum voltage (pu)"})
    box.show()


def plot_loading_distribution(df):
    fig = px.histogram(df, x="max_loading_percent", nbins=50, title="Maximum Line Loading Distribution", labels={"max_loading_percent": "Maximum loading (%)"})
    fig.update_layout(bargap=0.02)
    fig.show()


def plot_switching_configurations(df):
    counts = df["switch_config_key"].value_counts()
    print(f"Number of unique switching configurations: {len(counts):,}")
    print("Top 10 most frequent configurations:")
    display(counts.head(10).rename("count").to_frame())

    top10 = counts.head(10).reset_index()
    top10.columns = ["switch_config_key", "count"]
    fig = px.bar(top10, x="switch_config_key", y="count", title="Top 10 Switching Configurations", labels={"switch_config_key": "Switch configuration", "count": "Frequency"})
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()


def plot_load_scaling_distribution(load_scale_series):
    fig = px.histogram(load_scale_series.to_frame(), x="load_scaling_factor", nbins=50, title="Load Scaling Factor Distribution", labels={"load_scaling_factor": "Load perturbation factor"})
    fig.update_layout(bargap=0.02)
    fig.show()


def plot_correlation_heatmap(df):
    columns = ["total_load_mw", "total_loss_kw", "min_voltage_pu", "max_loading_percent"]
    corr = df[columns].corr()
    fig = go.Figure(data=go.Heatmap(
        z=corr.values,
        x=corr.columns,
        y=corr.index,
        colorscale="RdBu",
        zmin=-1,
        zmax=1,
        colorbar={"title": "Correlation"},
    ))
    fig.update_layout(title="Correlation Heatmap: Load, Loss, Voltage, Loading")
    fig.show()


def plot_constraint_violation_statistics(df):
    fig = px.histogram(df, x="violations", nbins=max(10, int(df["violations"].max()) + 1 if len(df) else 10), title="Constraint Violations per Sample", labels={"violations": "Voltage + overload violations"})
    fig.update_layout(bargap=0.02)
    fig.show()

In [ ]:
analysis_df, load_scale_series, dataset_attrs = collect_dataset_analysis(H5_PATH)

print_feasibility_report(dataset_attrs)

if len(analysis_df) == 0:
    raise RuntimeError("No valid samples were generated; analysis cannot continue.")

plot_loss_distribution(analysis_df)
plot_voltage_distribution(analysis_df)
plot_loading_distribution(analysis_df)
plot_switching_configurations(analysis_df)
plot_load_scaling_distribution(load_scale_series)
plot_correlation_heatmap(analysis_df)
plot_constraint_violation_statistics(analysis_df)

analysis_df.describe(include="all")

## 8. Dataset Summary for Graph Transformer Training

In [ ]:
def print_dataset_summary(h5_path=H5_PATH):
    with h5py.File(h5_path, "r") as h5:
        sample_names = sorted(name for name in h5.keys() if name.startswith("sample_"))
        if len(sample_names) == 0:
            raise RuntimeError("HDF5 file contains no sample groups.")

        first = h5[sample_names[0]]
        node_shape = first["node_features"].shape
        edge_shape = first["edge_features"].shape
        global_shape = first["global_features"].shape
        label_shape = first["labels"].shape

        print("Dataset Ready for Graph Transformer Training")
        print("=" * 52)
        print(f"HDF5 path          : {h5_path}")
        print(f"Number of samples  : {len(sample_names):,}")
        print(f"Node features      : shape={node_shape}, names={json.loads(h5.attrs['node_feature_names'])}")
        print(f"Edge features      : shape={edge_shape}, names={json.loads(h5.attrs['edge_feature_names'])}")
        print(f"Global features    : shape={global_shape}, names={json.loads(h5.attrs['global_feature_names'])}")
        print(f"Labels             : shape={label_shape}, names={json.loads(h5.attrs['label_names'])}")
        print("\nLabel definitions")
        print("- total_loss_kw: total active branch loss in kW")
        print("- min_voltage_pu: minimum bus voltage magnitude in per-unit")
        print("- max_loading_percent: maximum line loading percentage")
        print("- violations: voltage-limit violations plus line overloads")
        print("\nHDF5 storage structure")
        print("/sample_000001/")
        print("    node_features      float32 [num_buses, 4]")
        print("    edge_features      float32 [num_lines, 3]")
        print("    global_features    float32 [3]")
        print("    labels             float32 [4]")
        print("    metadata           JSON string")


print_dataset_summary(H5_PATH)